# Phase 2: Unsupervised Learning - Clustering Analysis

## 1. Clustering Rationale & Algorithm Selection

### 1.1 Rationale
The primary goal of applying unsupervised learning to the Clothing Fit Dataset is to move beyond predefined labels and discover latent patterns within the data. By grouping users based on physical attributes (height, weight, age) and product categories, we can:

* **Identify Body Type Clusters**: Group users with similar body profiles to provide more personalized size recommendations.
* **Enhance the Advice System**: Use these discovered patterns to refine the generative AI's explanations, making them more context-aware.
* **Uncover Latent User Preferences**: Clustering enables the identification of distinct user segments based on subtle fitting trends (e.g., a preference for oversized vs. snug fits) that are not explicitly captured by the dataset's original labels.


### 1.2 Algorithm Selection
For this analysis, we have selected the following two algorithms:

1. **K-Means Clustering**: 
   - **Why?** It is highly efficient for large datasets like ours and provides clear centroids that represent the 'average' user in each cluster.
   - **Suitability**:Our dataset contains hundreds of thousands of records, making K-Means the ideal choice due to its computational efficiency. Since many body measurements (like height and weight) often follow a normal distribution, K-Means is particularly effective at partitioning these users into distinct, interpretable size-based segments.

2. **DBSCAN (Density-Based Spatial Clustering of Applications with Noise)**:
   - **Why?** It is excellent at identifying clusters of arbitrary shapes and, crucially, it can detect and isolate outliers (noise).
   - **Suitability**:Clothing datasets often suffer from "noisy" data, such as unrealistic height/weight values entered by users. DBSCAN is uniquely suited for our project because it can automatically filter out these outliers instead of forcing them into a cluster, ensuring that our final recommendations are based on clean, high-density patterns of real body types.

## 2. Data Preparation
In this section, we prepare the preprocessed dataset for clustering by ensuring that the features are in a format that distance-based algorithms can process effectively.
### 2.1 Target removal

In [2]:
import pandas as pd
# Load the preprocessed data from Phase 1
df = pd.read_csv('Dataset/preprocessed_data.csv')
# Data Preparation: Target Removal 
# We remove 'fit' (target) and non-numeric review columns
X = df.drop(columns=['fit', 'review_summary', 'review_text'])
print(f"Features for clustering: {X.columns.tolist()}")

Features for clustering: ['size', 'quality', 'cup size', 'hips', 'bra size', 'height', 'length', 'cat_bottoms', 'cat_dresses', 'cat_new', 'cat_outerwear', 'cat_sale', 'cat_tops', 'cat_wedding', 'size_scaled', 'quality_scaled', 'hips_scaled', 'bra size_scaled', 'height_scaled', 'cup size_scaled', 'length_scaled']


we have to transate to Unsupervised Learning where the model explores the data's natural structure. To achieve this, we must remove the target variable fit to prevent the model from using pre-defined labels during the clustering process. We also exclude textual fields like review_text because our shosen algorithms K-Means and DBSCAN require numerical data to calculate Euclidean distances.

### 2.2 Feature Scaling

In [3]:
from sklearn.preprocessing import StandardScaler

# Initialize and apply the scaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)
print("Scaling completed. Data is ready for cluster determination.")
X_scaled_df.head()

Scaling completed. Data is ready for cluster determination.


,size,quality,cup size,hips,bra size,height,length,cat_bottoms,cat_dresses,cat_new,...,cat_sale,cat_tops,cat_wedding,size_scaled,quality_scaled,hips_scaled,bra size_scaled,height_scaled,cup size_scaled,length_scaled
0,-0.685281,1.058979,0.267479,-0.396924,-0.635700,0.294655,-0.165223,-0.475481,-0.539231,1.689038,...,-0.177329,-0.571148,-0.05773,-0.685281,1.058979,-0.396924,-0.635700,0.294655,0.267479,-0.165223
1,0.040384,-0.956397,-1.039065,-2.050813,0.008324,-1.241617,-0.165223,-0.475481,-0.539231,1.689038,...,-0.177329,-0.571148,-0.05773,0.040384,-0.956397,-2.050813,0.008324,-1.241617,-1.039065,-0.165223
2,-0.685281,-1.964085,-1.039065,-0.190188,-1.279724,0.678723,1.504556,-0.475481,-0.539231,1.689038,...,-0.177329,-0.571148,-0.05773,-0.685281,-1.964085,-0.190188,-1.279724,0.678723,-1.039065,1.504556
3,1.007937,1.058979,0.920751,-0.190188,0.008324,-0.089413,-0.165223,-0.475481,-0.539231,1.689038,...,-0.177329,-0.571148,-0.05773,1.007937,1.058979,-0.190188,0.008324,-0.089413,0.920751,-0.165223
4,0.645104,1.058979,-1.039065,-0.190188,0.008324,-1.241617,1.504556,-0.475481,-0.539231,1.689038,...,-0.177329,-0.571148,-0.05773,0.645104,1.058979,-0.190188,0.008324,-1.241617,-1.039065,1.504556


Since K-Means and DBSCAN are distance-based algorithms, features with different scales (e.g., height vs quality) must be normalized. We apply StandardScaler to ensure each feature contributes equally to the distance calculation.